# Why route at all? [Step 05.01 - Deciding before you spend]

> **MLCourse - Agentic AI - Agent Patterns**

Every agent you have built so far answers the same way: the request arrives, and
the big model is called. That is simple, and it is also the reason your bill and
your p95 latency look the way they do.

**Semantic routing** puts one cheap decision in front of that call:

```
user request
     |
     v
 [ EMBED ]  ~5 ms, no API call, no tokens
     |
     v
 compare against a handful of example utterances per route
     |
     +--> greeting      -> canned string        (0 tokens)
     +--> shipping_faq  -> stored FAQ answer    (0 tokens)
     +--> order_status  -> database lookup      (0 tokens)
     +--> arithmetic    -> Python               (0 tokens)
     +--> (nothing matched confidently) -> THE BIG MODEL
```

The interesting property is that the router itself is **not a model call**. It is
a dot product against a few dozen precomputed vectors. So the decision is
essentially free, and every request it diverts is a call you never pay for.

### What you'll learn

- Why an *embedding* router is different in kind from an *LLM* router, and how much
  cheaper it measurably is.
- How similarity to example utterances stands in for a trained classifier.
- Where routing goes wrong: it matches **topic**, not **intent**.

### Key takeaways

- Routing is a **precision instrument for cost**, not for quality. It does not make
  answers better; it stops you paying for answers you already have.
- An LLM router costs roughly as much as the call it is supposed to be protecting.
  Measure this before you build one.
- The router's mistakes are silent and land in the *handler*, not in the router -
  which is why the fallback route (notebook 03) matters more than the thresholds do.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


### The embedding model


In [ ]:
# `all-MiniLM-L6-v2` is 22M parameters, runs on CPU, and encodes a short sentence
# in single-digit milliseconds. That speed is the whole point: routing has to be
# cheap enough that it is obviously worth doing before the expensive call.

from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("all-MiniLM-L6-v2")


def embed(texts):
    """Encode a list of strings into L2-normalised vectors.

    Normalising means the dot product IS the cosine similarity, so every score
    below lives in [-1, 1] and is directly comparable.
    """
    return encoder.encode(list(texts), normalize_embeddings=True)


v = embed(["hello there", "hi!", "what is the capital of France"])
print("vector shape :", v.shape)
print("hello/hi     : %.3f" % float(v[0] @ v[1]))
print("hello/capital: %.3f" % float(v[0] @ v[2]))


### 1. The route set

A route set is just a dictionary. Each key is a route name; each value is a small
list of things a user might actually say to reach it.

This is the part beginners over-think. You do not need a labelled dataset, and you
do not need many examples - five short, *genuinely varied* utterances per route is
a working starting point. Varied matters far more than numerous: five paraphrases
of the same sentence give you one example, not five.

### The route set


In [ ]:
# A ROUTE is: a name, a handful of EXAMPLE UTTERANCES, and a handler.
# Nothing more. There is no training step and no classifier to fit.

ROUTES = {
    "greeting": [
        "hi there",
        "hello",
        "hey, good morning",
        "yo",
        "good evening",
    ],
    "shipping_faq": [
        "how long does delivery take",
        "when will my parcel arrive",
        "do you ship internationally",
        "what are your shipping costs",
        "how fast is standard delivery",
    ],
    "order_status": [
        "where is order 1042",
        "track my order 88",
        "what happened to order number 7",
        "status of order 1042 please",
        "has order 88 shipped yet",
    ],
    "arithmetic": [
        "what is 18 times 24",
        "compute 145 plus 92",
        "calculate 900 divided by 12",
        "how much is 37 minus 19",
        "multiply 13 by 7",
    ],
}

ROUTE_NAMES = list(ROUTES)
print("%d routes, %d example utterances total"
      % (len(ROUTES), sum(len(v) for v in ROUTES.values())))


### 2. The router

Encode every example utterance once, up front. At request time, encode the request
and take the dot product against all of them.

Two scoring choices exist, and they behave differently:

- **max similarity** - the route's score is its single best-matching example.
  Sensitive: one good example can carry a route. Also noisy.
- **centroid similarity** - average the route's example vectors into one vector
  and compare against that. Smoother, but it blurs routes whose examples are
  genuinely spread out.

We start with max, and compare the two properly in notebook 02.

### Encode the route examples ONCE. In production this happens at startup, or is


In [ ]:
# loaded from disk. It never happens per-request.
t0 = time.time()
ROUTE_VECTORS = {name: embed(examples) for name, examples in ROUTES.items()}
build_ms = (time.time() - t0) * 1000
print("built route index for %d utterances in %.0f ms" % (
    sum(len(v) for v in ROUTES.values()), build_ms))


def route_scores(text):
    """Return {route_name: max cosine similarity to that route's examples}."""
    q = embed([text])[0]
    return {name: float((vecs @ q).max()) for name, vecs in ROUTE_VECTORS.items()}


def route(text):
    """Return (best_route, score)."""
    scores = route_scores(text)
    best = max(scores, key=scores.get)
    return best, scores[best]


for utterance in ["hey there", "when does my package get here",
                  "where is order 1042", "what is 18 times 24",
                  "explain the causes of the French Revolution"]:
    name, score = route(utterance)
    print("%-45s -> %-14s %.3f" % (utterance, name, score))


Look at the last line. "Explain the causes of the French Revolution" belongs to no
route, and the router still returns one - with a low score. **A router always
returns something.** Turning that low score into "I do not know, send it to the
model" is the job of the confidence threshold, and it is the subject of notebook 03.

### 3. What routing actually costs

Now the measurement that justifies the whole pattern. We time the embedding router
on a batch of requests, then time an **LLM router** - the obvious alternative,
where you ask the model itself which route a request belongs to.

In [5]:
PROBE = [
    "hello there",
    "how long does shipping take to Spain",
    "track order 88 for me",
    "calculate 900 divided by 12",
    "who wrote Pride and Prejudice",
    "good evening",
]

# --- (a) embedding router -----------------------------------------------------
t0 = time.time()
emb_decisions = [route(p) for p in PROBE]
emb_seconds = time.time() - t0

print("embedding router: %d requests in %.3f s  (%.1f ms each)"
      % (len(PROBE), emb_seconds, emb_seconds / len(PROBE) * 1000))
print("tokens billed   : 0")

embedding router: 6 requests in 0.046 s  (7.7 ms each)
tokens billed   : 0


### (b) LLM router


In [ ]:
# Same job, done by asking the model. Note we are being generous to it: a tiny
# prompt, max_tokens=8, temperature 0.

classifier = make_llm(temperature=0.0, max_tokens=8)
CLASSIFY_PROMPT = (
    "Classify the user message into exactly one of these labels: "
    + ", ".join(ROUTE_NAMES) + ", other.\n"
    "Reply with the label only.\n\nMessage: %s"
)

llm_in = llm_out = 0
t0 = time.time()
llm_decisions = []
for p in PROBE:
    msg = safe_invoke(classifier, CLASSIFY_PROMPT % p)
    u = msg.usage_metadata or {}
    llm_in += u.get("input_tokens", 0)
    llm_out += u.get("output_tokens", 0)
    llm_decisions.append(msg.content.strip().splitlines()[-1].strip().lower())
llm_seconds = time.time() - t0
# subtract the pacing sleeps so the comparison is about the API, not our politeness
llm_seconds -= PACE * len(PROBE)

print("LLM router      : %d requests in %.3f s  (%.0f ms each, sleeps removed)"
      % (len(PROBE), llm_seconds, llm_seconds / len(PROBE) * 1000))
print("tokens billed   : %d in + %d out = %d" % (llm_in, llm_out, llm_in + llm_out))
print("cost            : $%.6f" % usd(llm_in, llm_out))


In [7]:
print("decisions, side by side")
print("-" * 76)
print("%-38s %-16s %-16s" % ("request", "embedding", "LLM"))
print("-" * 76)
for p, (e, s), l in zip(PROBE, emb_decisions, llm_decisions):
    print("%-38s %-16s %-16s" % (p[:37], "%s(%.2f)" % (e, s), l))
print("-" * 76)
speedup = llm_seconds / emb_seconds
print()
print("MEASURED: the embedding router was %.0fx faster and cost $0 instead of $%.6f"
      % (speedup, usd(llm_in, llm_out)))
print("Extrapolated to 1M requests, the LLM router alone would cost $%.2f"
      % (usd(llm_in, llm_out) / len(PROBE) * 1e6))

decisions, side by side
----------------------------------------------------------------------------
request                                embedding        LLM             
----------------------------------------------------------------------------
hello there                            greeting(0.82)   greeting        
how long does shipping take to Spain   shipping_faq(0.58) shipping_faq    
track order 88 for me                  order_status(0.93) order_status    
calculate 900 divided by 12            arithmetic(1.00) arithmetic      
who wrote Pride and Prejudice          order_status(0.03) other           
good evening                           greeting(1.00)   greeting        
----------------------------------------------------------------------------

MEASURED: the embedding router was 17x faster and cost $0 instead of $0.000103
Extrapolated to 1M requests, the LLM router alone would cost $17.09


### Reading that honestly

The LLM router is not *wrong* - it usually picks the same labels. It is simply
**the wrong tool for a decision that has to be free**. If you are going to spend a
model call deciding which model call to make, you have doubled your call count to
save at most one.

Where an LLM router does earn its place: routes that depend on reasoning rather
than topic ("does this request require a refund approval?"). Embeddings cannot see
that. Notebook 05.03 covers how to combine the two - cheap router first, model only
on the requests it is unsure about.

### 4. The failure mode you must internalise

Embeddings measure **topical similarity**, not **intent**. These two sentences are
near-neighbours in embedding space and want completely different handlers:

- "where is order 1042"        -> look up the order
- "I want to cancel order 1042" -> do NOT look up the order; start a cancellation

Let's watch it happen.

In [8]:
for probe in ["where is order 1042",
              "I want to cancel order 1042",
              "order 1042 arrived damaged, I need a refund"]:
    scores = route_scores(probe)
    ranked = sorted(scores.items(), key=lambda kv: -kv[1])
    print("%-46s %s" % (probe, "  ".join("%s=%.3f" % kv for kv in ranked[:3])))

where is order 1042                            order_status=1.000  shipping_faq=0.290  greeting=0.183
I want to cancel order 1042                    order_status=0.659  shipping_faq=0.207  arithmetic=0.170
order 1042 arrived damaged, I need a refund    order_status=0.677  shipping_faq=0.284  greeting=0.138


All three land on `order_status` with comparable confidence, because all three are
*about* an order. Only the first one wants the order-status handler.

Two defences, both used in practice:

1. **Add the missing route.** A `cancel_order` route with its own examples pulls the
   second sentence away. Routes you never wrote are the ones that hurt you.
2. **Keep handlers narrow and let them refuse.** A handler that can say "this is
   not what I do" turns a silent wrong answer into a fallback.

### Pitfalls

- **Routing to save money on a route you answer badly is a false economy.** Measure
  handler quality before you divert traffic to it.
- **Route example drift.** Real user phrasing moves; a route set written once and
  never revisited slowly loses recall. Log the fallback traffic and mine it.
- **Too many routes.** Past roughly a dozen, routes start colliding and you are
  better off with a two-level router (coarse route, then fine).
- **Case and punctuation matter less than you think, but length matters a lot.**
  A three-word request and a 300-word request do not embed comparably; consider
  routing on the last user turn only.

### Next

Notebook 02 builds a route set properly: centroid vs max scoring, measuring the
separation between routes, and what adding an example actually does.